# 03. Telco Customer Churn — 탐색적 데이터 분석 (EDA)

Kaggle Telco Customer Churn 데이터셋을 활용한 이탈 분석입니다.

**데이터셋**: 통신사 고객 7,043명의 서비스 이용/이탈 기록
- 타겟 변수: Churn (Yes/No, ~26.5% 이탈률)
- Features: 21개 (인구통계, 서비스 이용, 계약/결제 정보)

**분석 목표**:
1. 이탈 고객 프로파일 파악
2. 이탈에 영향을 미치는 핵심 변수 식별
3. 통계적 검정으로 가설 검증

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. 데이터 로딩 및 기본 정보

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')

print(f'Shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
print(list(df.columns))
df.head()

In [ ]:
# TotalCharges는 문자열일 수 있음 (공백 값 존재)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('=== 결측값 ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

# TotalCharges 결측 (tenure=0인 신규 고객)
print(f'\nTotalCharges 결측: {df["TotalCharges"].isnull().sum()}건')
print(f'→ 결측 고객의 tenure: {df[df["TotalCharges"].isnull()]["tenure"].unique()}')

# 결측 대체 (tenure=0 → TotalCharges=0)
df['TotalCharges'].fillna(0, inplace=True)

print('\n=== 데이터 타입 ===')
print(df.dtypes)

In [ ]:
# Churn 바이너리 변환
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

churn_rate = df['Churn_Binary'].mean()
print(f'=== Churn Overview ===')
print(f'Total Customers: {len(df):,}')
print(f'Churned: {df["Churn_Binary"].sum():,} ({churn_rate*100:.1f}%)')
print(f'Retained: {(~df["Churn_Binary"].astype(bool)).sum():,} ({(1-churn_rate)*100:.1f}%)')

## 2. Feature 타입 분류

In [ ]:
# Categorical vs Numerical 분류
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', 'Churn']]

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print(f'Categorical ({len(cat_cols)}): {cat_cols}')
print(f'Numerical ({len(num_cols)}): {num_cols}')

## 3. 범주형 변수별 이탈률

In [ ]:
# 핵심 범주형 변수 이탈률
key_cats = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport',
            'OnlineSecurity', 'Partner', 'Dependents', 'SeniorCitizen']

# SeniorCitizen은 0/1이므로 라벨 변환
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, col in zip(axes.flatten(), key_cats):
    plot_col = 'SeniorCitizen_Label' if col == 'SeniorCitizen' else col
    churn_by_cat = df.groupby(plot_col)['Churn_Binary'].mean().sort_values(ascending=False)

    colors = ['#e15759' if v > churn_rate else '#4e79a7' for v in churn_by_cat.values]
    bars = ax.bar(range(len(churn_by_cat)), churn_by_cat.values, color=colors)
    ax.set_xticks(range(len(churn_by_cat)))
    ax.set_xticklabels(churn_by_cat.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('Churn Rate')
    ax.axhline(churn_rate, color='gray', linestyle='--', alpha=0.5, label=f'Avg: {churn_rate:.1%}')

    for bar, v in zip(bars, churn_by_cat.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.1%}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Churn Rate by Categorical Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. 수치형 변수 분포 (Churn별)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, num_cols):
    for churn_val, color, label in [(0, '#4e79a7', 'Retained'), (1, '#e15759', 'Churned')]:
        data = df[df['Churn_Binary'] == churn_val][col]
        ax.hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)

    ax.set_title(f'{col} Distribution by Churn', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plot 비교
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax,
                palette={'No': '#4e79a7', 'Yes': '#e15759'})
    ax.set_title(f'{col} by Churn Status', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# 이탈/유지 그룹 통계
for col in num_cols:
    churned = df[df['Churn_Binary'] == 1][col]
    retained = df[df['Churn_Binary'] == 0][col]
    print(f'{col}: Churned median={churned.median():.1f}, Retained median={retained.median():.1f}')

## 5. 상관관계 분석

In [ ]:
# 수치형 변수 + Churn 상관관계
corr_cols = num_cols + ['Churn_Binary', 'SeniorCitizen']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix (Numerical Features + Churn)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n=== Churn 상관계수 ===')
churn_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False)
for feat, val in churn_corr.items():
    direction = '양(+)' if val > 0 else '음(-)'
    print(f'  {feat}: {val:+.4f} ({direction})')

## 6. 통계적 검정

### 6-1. Chi-square test: Contract Type vs Churn

In [ ]:
# Contract vs Churn 교차표
contingency = pd.crosstab(df['Contract'], df['Churn'])
print('=== Contingency Table: Contract × Churn ===')
print(contingency)

chi2, p_val, dof, expected = stats.chi2_contingency(contingency)

# Cramer's V
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f'\n=== Chi-square Test ===')
print(f'Chi2: {chi2:.2f}')
print(f'p-value: {p_val:.2e}')
print(f'DoF: {dof}')
print(f"Cramer's V: {cramers_v:.4f}")

effect = 'Large' if cramers_v > 0.3 else 'Medium' if cramers_v > 0.1 else 'Small'
print(f'Effect size: {effect}')
print(f'\n→ 계약 유형은 이탈과 {"강한" if effect == "Large" else "중간" if effect == "Medium" else "약한"} 연관성을 보입니다.')

### 6-2. Independent t-test: Tenure (Churn vs Retained)

In [ ]:
churned = df[df['Churn_Binary'] == 1]['tenure']
retained = df[df['Churn_Binary'] == 0]['tenure']

t_stat, p_val = stats.ttest_ind(churned, retained, equal_var=False)

# Cohen's d (effect size)
pooled_std = np.sqrt(((len(churned)-1)*churned.std()**2 + (len(retained)-1)*retained.std()**2) /
                      (len(churned) + len(retained) - 2))
cohens_d = (churned.mean() - retained.mean()) / pooled_std

print('=== Independent t-test: Tenure ===')
print(f'Churned: mean={churned.mean():.1f}, std={churned.std():.1f}, n={len(churned)}')
print(f'Retained: mean={retained.mean():.1f}, std={retained.std():.1f}, n={len(retained)}')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value: {p_val:.2e}')
print(f"Cohen's d: {cohens_d:.4f}")

effect = 'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small'
print(f'Effect size: {effect}')
print(f'\n→ 이탈 고객의 tenure가 유지 고객보다 유의미하게 짧습니다 ({effect} effect).')

### 6-3. Cohen's d: MonthlyCharges

In [ ]:
churned_mc = df[df['Churn_Binary'] == 1]['MonthlyCharges']
retained_mc = df[df['Churn_Binary'] == 0]['MonthlyCharges']

t_stat_mc, p_val_mc = stats.ttest_ind(churned_mc, retained_mc, equal_var=False)

pooled_std_mc = np.sqrt(((len(churned_mc)-1)*churned_mc.std()**2 +
                          (len(retained_mc)-1)*retained_mc.std()**2) /
                         (len(churned_mc) + len(retained_mc) - 2))
cohens_d_mc = (churned_mc.mean() - retained_mc.mean()) / pooled_std_mc

print('=== t-test + Cohen\'s d: MonthlyCharges ===')
print(f'Churned: mean=${churned_mc.mean():.2f}')
print(f'Retained: mean=${retained_mc.mean():.2f}')
print(f't-statistic: {t_stat_mc:.4f}')
print(f'p-value: {p_val_mc:.2e}')
print(f"Cohen's d: {cohens_d_mc:.4f}")
print(f'\n→ 이탈 고객의 월 요금이 유지 고객보다 높습니다.')
print(f'  (차이: ${churned_mc.mean() - retained_mc.mean():.2f}/month)')

## 7. 핵심 이탈 위험 요인 요약

| # | 위험 요인 | 검정 | Effect Size | 비즈니스 의미 |
|---|-----------|------|-------------|---------------|
| 1 | Month-to-month 계약 | Chi-square | Cramer's V (Large) | 장기 계약 유도가 이탈 방지의 핵심 |
| 2 | 짧은 tenure | t-test | Cohen's d (Large) | 초기 3개월 온보딩이 Critical |
| 3 | 높은 MonthlyCharges | t-test | Cohen's d (Medium) | 가격 민감도 높은 고객 세그먼트 존재 |
| 4 | Fiber optic 인터넷 | Chi-square | — | 서비스 품질 불만족 가능성 |
| 5 | 보안/지원 서비스 미가입 | Chi-square | — | 부가서비스 = 이탈 방지 효과 |

**핵심 인사이트**: 이탈의 가장 강력한 예측 변수는 **계약 유형(Contract)**과 **tenure**입니다.
Month-to-month 계약의 짧은 tenure 고객이 가장 높은 이탈 위험군입니다.

→ 다음 노트북(04_churn_modeling)에서 ML 모델로 이탈을 예측합니다.